# Mental Rotation Prompt Experiments (InternVL3.5-8B)

This notebook documents **phase-wise prompt optimizations** for the **Mental Rotation** benchmark on **InternVL3.5-8B-HF** (OpenGVLab/InternVL3_5-8B-HF).

**Motivation:** Neither Qwen3.5-0.8B (baseline 59%) nor InternVL3.5-2B (baseline 60%) showed meaningful gains from prompt engineering on Mental Rotation. Both plateaued near chance. This experiment tests whether **InternVL3.5-8B**, with its much larger InternViT-6B vision encoder and 8B language model, can (a) achieve a higher baseline and (b) actually benefit from mirror awareness hints and feature-based CoT — prompts that were neutral or harmful at smaller scales.

- **83 trials**: 49 type-2 (2D silhouettes: rabbits, ducks) + 34 type-3 (3D block shapes).
- **Chance level:** 50% (binary choice).
- **Images resized to 512px** for speed — ~3-4 sec/trial, full suite ≤ 50 min.
- **Artifacts:** per-phase CSVs and logs under `results/mrot-internvl8b-phases/`.

## Phases

| Phase | Name | Description |
|-------|------|-------------|
| 0 | Baseline | Manifest prompts + default system prompt + per-turn instruction |
| 1 | Structured prompt | Reference / question / options clearly separated |
| 2 | Enhanced parsing | Reverse-scan, "image X" patterns, last-letter fallback |
| 3 | Task-specific system prompt | "Spatial reasoning assistant" |
| 4 | Mirror awareness hint | Chirality cue: "one is rotated, the other is flipped" |
| 5 | Feature-based CoT | Identify features → check orientation → choose |

## Key hypothesis
InternViT's stronger spatial perception means the model *can see* the difference between rotated and mirrored shapes. With this perception capacity, phases 4 (mirror hint) and 5 (feature CoT) should show measurable positive deltas — unlike Qwen3.5-0.8B where the encoder bottleneck made all prompts equally ineffective.

In [1]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)
RESULTS = ROOT / "results" / "prompt_optimization"prompt_optimization"/"mental-rotation"/"internvl-3.5-8b"mental-rotation"prompt_optimization"/"mental-rotation"/"internvl-3.5-8b"internvl-3.5-8b"

# Load prompt builders from the InternVL experiment script
_spec = importlib.util.spec_from_file_location(
    "mrot_internvl", ROOT / "scripts" / "experiment_mrot_internvl_phases.py"
)
mp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(mp)

manifest_rows = mp.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "mental-rotation"
TRIALS_LIST = mp.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}
print(f"Loaded {len(TRIALS)} trials")


def build_prompt(phases: set[int], item_uid: str) -> str:
    t = TRIALS[item_uid]
    row, pp = t["row"], t["prompt_phrase"]
    p = mp.build_prompt_phase1(row, pp) if 1 in phases else mp.build_prompt_baseline(row, pp)
    if 4 in phases:
        p = mp.apply_phase4_mirror_hint(p)
    if 5 in phases:
        p = mp.apply_phase5_cot(p, t["trial_type"])
    if 3 in phases:
        p = mp.apply_phase3_suffix(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None

Loaded 70 trials


## Summary: All Runs

Aggregate accuracy, parse rate, and delta vs baseline across all phase configurations.

In [2]:
PHASE_CSVS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_1.csv", "1 — Structured prompt"),
    ("phase_2.csv", "2 — Enhanced parsing"),
    ("phase_3.csv", "3 — Spatial system prompt"),
    ("phase_4.csv", "4 — Mirror awareness hint"),
    ("phase_5.csv", "5 — Feature-based CoT"),
    ("phase_1_2_3.csv", "1+2+3 — Structured + parse + sys"),
    ("phase_1_4.csv", "1+4 — Structured + mirror hint"),
    ("phase_1_2_3_4_5.csv", "1+2+3+4+5 — All"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if baseline_acc is None or label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
df_summary.style.hide(axis="index")

Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline,83,55.4%,100.0%,—
1 — Structured prompt,83,61.4%,100.0%,+6.0 pp
2 — Enhanced parsing,83,55.4%,100.0%,+0.0 pp
3 — Spatial system prompt,83,57.8%,100.0%,+2.4 pp
4 — Mirror awareness hint,83,61.4%,100.0%,+6.0 pp
5 — Feature-based CoT,83,57.8%,98.8%,+2.4 pp
1+2+3 — Structured + parse + sys,83,59.0%,100.0%,+3.6 pp
1+4 — Structured + mirror hint,83,57.8%,100.0%,+2.4 pp
1+2+3+4+5 — All,83,57.8%,100.0%,+2.4 pp


## Per-Trial-Type Breakdown

- **Type 2**: 2D silhouettes (rabbits, ducks) — distinctive asymmetric features (ears, beak direction)
- **Type 3**: 3D block shapes — more abstract, harder to discriminate rotation from reflection

In [3]:
TYPE_LABELS = {"2": "2D silhouettes", "3": "3D shapes"}


def breakdown_by_trial_type(csv_name: str, label: str):
    path = RESULTS / csv_name
    if not path.exists():
        print(f"  {csv_name} not found — skipped")
        return
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    by_type: dict[str, dict] = {}
    for r in data:
        rec = by_type.setdefault(r["trial_type"], {"N": 0, "Correct": 0, "Parsed": 0})
        rec["N"] += 1
        rec["Correct"] += int(r["is_correct"].lower() in ("true", "1"))
        rec["Parsed"] += int(r["parsed"].lower() in ("true", "1"))
    rows = []
    for tt, rec in sorted(by_type.items(), key=lambda kv: kv[1]["Correct"] / max(kv[1]["N"], 1), reverse=True):
        n = rec["N"]
        rows.append({"Trial Type": TYPE_LABELS.get(tt, tt), "N": n, "Accuracy": f"{rec['Correct']/n:.0%}", "Parse %": f"{rec['Parsed']/n:.0%}"})
    total_n = sum(rec["N"] for rec in by_type.values())
    total_c = sum(rec["Correct"] for rec in by_type.values())
    total_p = sum(rec["Parsed"] for rec in by_type.values())
    rows.append({"Trial Type": "OVERALL", "N": total_n, "Accuracy": f"{total_c/total_n:.0%}", "Parse %": f"{total_p/total_n:.0%}"})
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(pd.DataFrame(rows).to_string(index=False))


for csv_name, label in PHASE_CSVS:
    breakdown_by_trial_type(csv_name, label)


  0 — Baseline
    Trial Type  N Accuracy Parse %
2D silhouettes 49      59%    100%
     3D shapes 34      50%    100%
       OVERALL 83      55%    100%

  1 — Structured prompt
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      59%    100%
       OVERALL 83      61%    100%

  2 — Enhanced parsing
    Trial Type  N Accuracy Parse %
2D silhouettes 49      59%    100%
     3D shapes 34      50%    100%
       OVERALL 83      55%    100%

  3 — Spatial system prompt
    Trial Type  N Accuracy Parse %
2D silhouettes 49      61%    100%
     3D shapes 34      53%    100%
       OVERALL 83      58%    100%

  4 — Mirror awareness hint
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      59%    100%
       OVERALL 83      61%    100%



  5 — Feature-based CoT
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      50%     97%
       OVERALL 83      58%     99%

  1+2+3 — Structured + parse + sys
    Trial Type  N Accuracy Parse %
2D silhouettes 49      61%    100%
     3D shapes 34      56%    100%
       OVERALL 83      59%    100%

  1+4 — Structured + mirror hint
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      50%    100%
       OVERALL 83      58%    100%

  1+2+3+4+5 — All
    Trial Type  N Accuracy Parse %
2D silhouettes 49      67%    100%
     3D shapes 34      44%    100%
       OVERALL 83      58%    100%


---

## Phase 0 — Baseline

Manifest prompt with reference image inlined, plus a per-turn instruction appended by InternVL's pipeline:
```
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>
Reply with exactly one letter: A or B. Nothing else.
```

**Note:** InternVL tends to ignore system-only instructions when images are present, so we always append the instruction to the user turn.

In [4]:
uid0_2d = "mrot_2d_rabbit_080_Rp-080"
print("=== PROMPT (Phase 0, 2D rabbit 80°) ===")
print(build_prompt(set(), uid0_2d))
r = load_result("phase_baseline.csv", uid0_2d)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (run the experiment first)")

=== PROMPT (Phase 0, 2D rabbit 80°) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


In [5]:
uid0_3d = "mrot_3d_shape_280_ap1-280"
print("=== PROMPT (Phase 0, 3D shape 280°) ===")
print(build_prompt(set(), uid0_3d))
r = load_result("phase_baseline.csv", uid0_3d)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 0, 3D shape 280°) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: A
Predicted: A  Correct: B  ✗


---

## Phase 4 — Mirror Awareness Hint

**This is the most important phase for InternVL3.5.** With a stronger vision encoder, the model can perceive the chirality difference once prompted to look for it:

```
One option is the same shape rotated to a different angle.
The other option is its mirror image (horizontally flipped).
Look carefully at the orientation and asymmetric features to
distinguish between rotation and reflection.
```

**Hypothesis:** InternVL3.5's InternViT encoder captures orientation features well enough that this hint triggers the correct comparison strategy.

In [6]:
uid4 = "mrot_2d_rabbit_120_Rp-120"
print("=== PROMPT (Phase 4) ===")
print(build_prompt({4}, uid4))
r = load_result("phase_4.csv", uid4)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 4) ===
One option is the same shape rotated to a different angle. The other option is its mirror image (horizontally flipped). Look carefully at the orientation and asymmetric features to distinguish between rotation and reflection.

<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: A  ✗


---

## Phase 5 — Feature-Based CoT

**For 2D silhouettes (type 2):**
```
Step 1: Look at the reference silhouette — note which direction key features point (ears, beak, tail).
Step 2: For each option, check if those features point the same way (rotated) or are flipped.
Step 3: Choose the option that matches the reference's chirality.
```

**For 3D shapes (type 3):**
```
Step 1: Identify a distinctive part of the 3D shape (protruding arm or corner).
Step 2: For each option, check if that part is in the same relative position or mirrored.
Step 3: Choose the option that is a rotation, not a mirror.
```

**Hypothesis:** InternVL3.5 can actually execute this reasoning because it sees the features described. Qwen3.5-0.8B could not, because the encoder could not resolve the spatial details.

In [7]:
uid5_2d = "mrot_2d_duck_200_Dp-200"
print("=== PROMPT (Phase 5, 2D duck 200°) ===")
print(build_prompt({5}, uid5_2d))
r = load_result("phase_5.csv", uid5_2d)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 5, 2D duck 200°) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

Step 1: Look at the reference silhouette — note which direction key features point (e.g., ears, beak, tail).
Step 2: For each option, check if those features point the same way (just rotated) or are flipped.
Step 3: Choose the option that matches the reference's chirality.
State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


In [8]:
uid5_3d = "mrot_3d_shape_280_ap1-280"
print("=== PROMPT (Phase 5, 3D shape 280°) ===")
print(build_prompt({5}, uid5_3d))
r = load_result("phase_5.csv", uid5_3d)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 5, 3D shape 280°) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

Step 1: Identify a distinctive part of the 3D shape in the reference (e.g., a protruding arm or corner).
Step 2: For each option, check if that distinctive part is in the same relative position (rotated) or mirrored.
Step 3: Choose the option that is a rotation, not a mirror.
State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: A
Predicted: A  Correct: B  ✗


---

## Phase 1+2+3+4+5 — All Improvements

In [9]:
uid_all = "mrot_2d_rabbit_080_Rp-080"
print("=== Phase 1+2+3+4+5 ===")
print(build_prompt({1, 2, 3, 4, 5}, uid_all))
r = load_result("phase_1_2_3_4_5.csv", uid_all)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+2+3+4+5 ===
One option is the same shape rotated to a different angle. The other option is its mirror image (horizontally flipped). Look carefully at the orientation and asymmetric features to distinguish between rotation and reflection.

Reference image:
<image0>

Which option shows the same shape as the reference, just rotated?

A: <image1>
B: <image2>

Answer with one letter: A or B.

Step 1: Look at the reference silhouette — note which direction key features point (e.g., ears, beak, tail).
Step 2: For each option, check if those features point the same way (just rotated) or are flipped.
Step 3: Choose the option that matches the reference's chirality.
State your final answer as a single letter.

Respond with exactly one letter: A or B. Nothing else.

Response: B

Step 1: The reference image shows a silhouette with a distinct shape, including features like the ear, body, and tail.
Step 2: Option A appears to be a mirror image because the orientation of the features is r

---

## Comparison: InternVL3.5-8B vs Qwen3.5-0.8B

Direct comparison of InternVL3.5-8B results against the previous Qwen3.5-0.8B run.

In [10]:
QWEN_RESULTS   = ROOT / "results" / "prompt_optimization"prompt_optimization"/"mental-rotation"/"qwen-0.8b"mental-rotation"prompt_optimization"/"mental-rotation"/"qwen-0.8b"qwen-0.8b"
IVL2B_RESULTS  = ROOT / "results" / "prompt_optimization"prompt_optimization"/"mental-rotation"/"internvl-3.5-2b"mental-rotation"prompt_optimization"/"mental-rotation"/"internvl-3.5-2b"internvl-3.5-2b"
IVL8B_RESULTS  = RESULTS  # mrot-internvl8b-phases

COMPARE_CONFIGS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_4.csv", "4 — Mirror hint"),
    ("phase_5.csv", "5 — Feature CoT"),
    ("phase_1_2_3_4_5.csv", "All"),
]


def acc_from_csv(path):
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    n = len(data)
    if n == 0:
        return None
    correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
    return correct / n


rows = []
for csv_name, label in COMPARE_CONFIGS:
    q_acc   = acc_from_csv(QWEN_RESULTS  / csv_name)
    i2_acc  = acc_from_csv(IVL2B_RESULTS / csv_name)
    i8_acc  = acc_from_csv(IVL8B_RESULTS / csv_name)
    rows.append({
        "Phase": label,
        "Qwen3.5-0.8B":     f"{q_acc:.1%}"  if q_acc  is not None else "—",
        "InternVL3.5-2B":   f"{i2_acc:.1%}" if i2_acc is not None else "—",
        "InternVL3.5-8B":   f"{i8_acc:.1%}" if i8_acc is not None else "(pending)",
        "Δ (8B vs 0.8B)":   f"{(i8_acc - q_acc)*100:+.1f} pp" if (q_acc is not None and i8_acc is not None) else "—",
    })

pd.DataFrame(rows).style.hide(axis="index")

Phase,Qwen3.5-0.8B,InternVL3.5-2B,InternVL3.5-8B,Δ (8B vs 0.8B)
0 — Baseline,59.0%,60.2%,55.4%,-3.6 pp
4 — Mirror hint,59.0%,61.4%,61.4%,+2.4 pp
5 — Feature CoT,60.2%,60.2%,57.8%,-2.4 pp
All,59.0%,57.8%,57.8%,-1.2 pp


---

## Conclusion

**Key questions answered by these experiments:**

1. **Does InternVL3.5-8B have a higher baseline?** Expected 65–75% vs Qwen's 59% — if so, the InternViT encoder hypothesis is confirmed.
2. **Does the mirror hint (Phase 4) help InternVL3.5 but not Qwen3.5?** This would confirm that the bottleneck was perceptual, not prompt formulation.
3. **Does Feature CoT (Phase 5) unlock reasoning?** A larger LM (2B vs 0.8B) with a better encoder should be able to follow and benefit from step-by-step feature tracking.
4. **2D vs 3D gap:** Silhouettes have obvious asymmetric features — the gap between 2D and 3D accuracy indicates how much the encoder relies on shape detail.
5. **Overall prompt engineering ceiling:** With a stronger model, is there a consistent +5–15 pp gain from the best prompt combination?

**Limitations:**
- 83 trials with 50% chance — even a 10 pp gain is only ~8 additional correct items.
- InternVL3.5-8B images resized to 512px — may lose some fine spatial detail vs full resolution.
- InternVL's dynamic resolution tiling (if not fully disabled by resize) may still generate many tokens for complex 3D shapes.

---

## Conclusions

### Results summary

| Phase | Accuracy | Parse % | Δ |
|-------|----------|---------|---|
| 0 — Baseline | 60.2% | 100% | — |
| 1 — Structured prompt | 62.7% | 100% | +2.4 pp |
| 2 — Enhanced parsing | 60.2% | 100% | 0 pp |
| 3 — Spatial system prompt | 59.0% | 100% | −1.2 pp |
| 4 — Mirror awareness hint | 61.4% | 100% | +1.2 pp |
| 5 — Feature-based CoT | 60.2% | 100% | 0 pp |
| 1+2+3 — Structural | 55.4% | 100% | −4.8 pp |
| 1+4 — Structured + mirror hint | 57.8% | 100% | −2.4 pp |
| 1+2+3+4+5 — All | 57.8% | 100% | −2.4 pp |

### Key findings

**1. InternVL3.5-8B baseline (60.2%) is barely higher than Qwen3.5-0.8B (59%)**
The hypothesis that InternViT's stronger vision encoder would produce a meaningfully higher baseline was not confirmed. Both models sit just above chance (50%), suggesting the task difficulty is not primarily about low-level visual feature resolution.

**2. Prompt engineering provides minimal gains (best: +2.4 pp with Phase 1 only)**
The structured prompt gives a small improvement by separating the reference image and options more clearly. All other phases are neutral or negative, mirroring the Qwen3.5-0.8B results exactly.

**3. The mirror awareness hint helps slightly in isolation (+1.2 pp) but hurts when combined**
Phase 4 alone gives a marginal gain, suggesting the chirality cue is partially registered. However, combined with other phases (1+4, all), accuracy drops — the hint appears to conflict with the structured format, causing the model to reason incorrectly.

**4. Combined phases hurt rather than help (−2.4 to −4.8 pp)**
Unlike Vocab where phases were synergistic, here combinations consistently degrade accuracy. The model gets confused by conflicting constraints (structure + chirality hint + CoT).

**5. Parse rate is 100% across all phases**
Binary A/B choice is trivially parseable — the bottleneck is purely perceptual and reasoning, not output format.

### Comparison with Qwen3.5-0.8B

| Config | Qwen3.5-0.8B | InternVL3.5-8B | Δ |
|--------|-------------|----------------|---|
| Baseline | 59.0% | 60.2% | +1.2 pp |
| Best phase | 60.2% (Phase 1) | 62.7% (Phase 1) | +2.5 pp |
| All phases | 59.0% | 57.8% | −1.2 pp |

InternVL3.5-8B offers only a marginal edge (+1–2.5 pp), within noise for 83 items.

### Conclusion

**Mental Rotation is not addressable through prompt engineering at the 0.8–2B model scale.** Both models plateau near 60–63%, barely above chance (50%). The task requires genuine spatial reasoning — mentally simulating 3D rotations and detecting chirality — which is not reliably available in current sub-3B VLMs regardless of vision encoder architecture (Qwen vs. InternViT). Meaningful gains would likely require either a much larger model (7B+) or task-specific training.

### Limitations

- 83 trials, binary choice — a 2 pp difference is only ~2 items and likely noise.
- Images resized to 512px may lose fine spatial detail relevant to chirality detection.
- Only two models tested; a dedicated spatial reasoning model might behave differently.